"""

**Continuous Assessment 2**

Applied Statistics and Machine Learning

Student Name : Nandhitha Ganesan Rajambal; Student ID : 20069568

Student Name : Varsha Sundararaj; Student ID : 20065968

**Question:**

To build regression models (Linear Regression and Support Vector Regression) to predict the resolution time of customer support tickets

"""

# **1. Data Preparation**

# **Data Reading**

In [1]:
# ----------------------------------------------------
# 1. DATA PREPARATION
#-----------------------------------------------------

# importing libraries
from pandas import read_csv,get_dummies,DataFrame
import pandas as pd
# loading dataset
data=read_csv('/content/customer_support_tickets.csv')
data.head()
data.shape
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

**Explanation:**
- First,the dataset was loaded using read_csv().
- Checked the first few rows using head() to understand how the data looks.
- Used info() to check the no. of records, no. of columns, data types, and missing values.
- The dataset contains "8469 rows and 17 columns".
- Out of these 17 columns, some columns had missing values:

      - Resolution had 2769 non-null
      - First Response Time had 5650 non-null
      - Time to Resolution had 2769 non-null
      - Customer Satisfaction Rating had 2769 non-null
- Hence, the data cleaning is needed before moving to modelling.

# Data Cleaning

In [2]:
# filling missing values using mode
m1 = data['Resolution'].mode()[0]
data['Resolution'] = data['Resolution'].fillna(m1)

m2=data['First Response Time'].mode()[0]
data['First Response Time'] = data['First Response Time'].fillna(m2)

m3=data['Time to Resolution'].mode()[0]
data['Time to Resolution'] = data['Time to Resolution'].fillna(m3)

m4=data['Customer Satisfaction Rating'].mode()[0]
data['Customer Satisfaction Rating'] = data['Customer Satisfaction Rating'].fillna(m4)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    8469 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

**Explanation:**
- Some columns had missing values, so I handled them before applying any model.
- The missing values were filled using the mode method.
- After Filling

        - Resolution became 8469 non-null
        - First Response Time became 8469 non-null
        - Time to Resolution became 8469 non-null
        - Customer Satisfaction Rating became 8469 non-null
- This step is important because machine learning models cannot work properly if null values are present.

**Removing Unnecessary Columns**

In [3]:
# Drop unwanted columns
data=data.drop(['Ticket ID','Customer Name','Customer Email',
                'Ticket Subject','Ticket Description','Resolution'],axis=1)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer Age                  8469 non-null   int64  
 1   Customer Gender               8469 non-null   object 
 2   Product Purchased             8469 non-null   object 
 3   Date of Purchase              8469 non-null   object 
 4   Ticket Type                   8469 non-null   object 
 5   Ticket Status                 8469 non-null   object 
 6   Ticket Priority               8469 non-null   object 
 7   Ticket Channel                8469 non-null   object 
 8   First Response Time           8469 non-null   object 
 9   Time to Resolution            8469 non-null   object 
 10  Customer Satisfaction Rating  8469 non-null   float64
dtypes: float64(1), int64(1), object(9)
memory usage: 727.9+ KB


**Explanation:**
- Removed the unwanted columns - "Ticket ID,Customer Name, Customer Email,Ticket Subject,Ticket Description,Resolution", as these columns are not needed for prediction.
- After dropping them, the dataset size changed from 17 columns to 11 columns.

# Data Encoding

In [4]:
# Checking Unique Values

#Checking no.of unique values in each column
for col in data.columns:
    print(col, data[col].nunique())

Customer Age 53
Customer Gender 3
Product Purchased 42
Date of Purchase 730
Ticket Type 5
Ticket Status 3
Ticket Priority 4
Ticket Channel 4
First Response Time 5470
Time to Resolution 2728
Customer Satisfaction Rating 5


In [5]:
#one-hot encoding to categorical columns
#Customer Gender,Product Purchased,Ticket Type,Ticket Status, Ticket Priority, Ticket Channel are multicoded
data = get_dummies(data, columns=['Customer Gender','Product Purchased','Ticket Type','Ticket Status',
                                  'Ticket Priority','Ticket Channel'], dtype=int)

**Explanation:**
- Machine Learning models cannot directly understand categorical text values.
- Hence, converted the following categorical columns into numeric format using one-hot encoding - "Customer Gender,Product Purchased,Ticket Type,Ticket Status, Ticket Priority, Ticket Channel".
- After encoding, the total no. of columns increased from 11 to 66.
- The column got changed because the each category got converted into separate numeric columns.

**Date Convesrion**

In [6]:
#covert Date columns

#converting the object(date) columns into datetime format
data['Date of Purchase'] = pd.to_datetime(data['Date of Purchase'], errors='coerce')
data['First Response Time'] = pd.to_datetime(data['First Response Time'], errors='coerce')
data['Time to Resolution'] = pd.to_datetime(data['Time to Resolution'], errors='coerce')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 66 columns):
 #   Column                                            Non-Null Count  Dtype         
---  ------                                            --------------  -----         
 0   Customer Age                                      8469 non-null   int64         
 1   Date of Purchase                                  8469 non-null   datetime64[ns]
 2   First Response Time                               8469 non-null   datetime64[ns]
 3   Time to Resolution                                8469 non-null   datetime64[ns]
 4   Customer Satisfaction Rating                      8469 non-null   float64       
 5   Customer Gender_Female                            8469 non-null   int64         
 6   Customer Gender_Male                              8469 non-null   int64         
 7   Customer Gender_Other                             8469 non-null   int64         
 8   Product Purchased_Adobe Phot

**Explanation:**
- The date columns were orignally stored as object type.
- Then, Converted them into datetime format so that, it could perform time based calculations.
- After conversion:

      - Date of Purchase became datetime64[ns]
      - First Response Time became datetime64[ns]
      - Time to Resolution became datetime64[ns]
- This conversion is important because my regression target depends on time difference.

**Target Variable Creation**

In [7]:
# Create Target Variable(hours)
data['Resolution Hours'] = (data['Time to Resolution'] - data['First Response Time']).dt.total_seconds().abs() / 3600
data[['Resolution Hours']].head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 67 columns):
 #   Column                                            Non-Null Count  Dtype         
---  ------                                            --------------  -----         
 0   Customer Age                                      8469 non-null   int64         
 1   Date of Purchase                                  8469 non-null   datetime64[ns]
 2   First Response Time                               8469 non-null   datetime64[ns]
 3   Time to Resolution                                8469 non-null   datetime64[ns]
 4   Customer Satisfaction Rating                      8469 non-null   float64       
 5   Customer Gender_Female                            8469 non-null   int64         
 6   Customer Gender_Male                              8469 non-null   int64         
 7   Customer Gender_Other                             8469 non-null   int64         
 8   Product Purchased_Adobe Phot

**Explanation:**
- First, Created a new target varaiable called Resolution hours.
- This represents the no.of hours taken to resolve each support tickets.
- After Creating this variable, the total no.of columns increases from 66 to 67.
- Resolution Hours is the final target variable for this regression model.

**Feature Engineering**

In [8]:
# Create Extra Date Features

# extracting useful values from datetime columns
data['Purchase Month'] = data['Date of Purchase'].dt.month
data['Purchase Day'] = data['Date of Purchase'].dt.day
data['First Response Hour'] = data['First Response Time'].dt.hour
data['First Response Minute'] = data['First Response Time'].dt.minute
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 71 columns):
 #   Column                                            Non-Null Count  Dtype         
---  ------                                            --------------  -----         
 0   Customer Age                                      8469 non-null   int64         
 1   Date of Purchase                                  8469 non-null   datetime64[ns]
 2   First Response Time                               8469 non-null   datetime64[ns]
 3   Time to Resolution                                8469 non-null   datetime64[ns]
 4   Customer Satisfaction Rating                      8469 non-null   float64       
 5   Customer Gender_Female                            8469 non-null   int64         
 6   Customer Gender_Male                              8469 non-null   int64         
 7   Customer Gender_Other                             8469 non-null   int64         
 8   Product Purchased_Adobe Phot

**Explanation:**
- Extracted additional features from the date columns - "Purchase Month, Purchase Day, First Response Hour, First Response Minute".
- These features can help the model learn time-related patterns.
- After adding these columns, the no.of columns increased from 67 to 71 columns.

**Final Cleaning after Conversation**

In [9]:
# Filling the missing values after datetime conversion

#filling the missing values in newly created numeric columns
m5 = data['Resolution Hours'].median()
data['Resolution Hours'] = data['Resolution Hours'].fillna(m5)

m6 = data['Purchase Month'].median()
data['Purchase Month'] = data['Purchase Month'].fillna(m6)

m7 = data['Purchase Day'].median()
data['Purchase Day'] = data['Purchase Day'].fillna(m7)

m8 = data['First Response Hour'].median()
data['First Response Hour'] = data['First Response Hour'].fillna(m8)

m9 = data['First Response Minute'].median()
data['First Response Minute'] = data['First Response Minute'].fillna(m9)

m10 = data['Customer Satisfaction Rating'].median()
data['Customer Satisfaction Rating'] = data['Customer Satisfaction Rating'].fillna(m10)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 71 columns):
 #   Column                                            Non-Null Count  Dtype         
---  ------                                            --------------  -----         
 0   Customer Age                                      8469 non-null   int64         
 1   Date of Purchase                                  8469 non-null   datetime64[ns]
 2   First Response Time                               8469 non-null   datetime64[ns]
 3   Time to Resolution                                8469 non-null   datetime64[ns]
 4   Customer Satisfaction Rating                      8469 non-null   float64       
 5   Customer Gender_Female                            8469 non-null   int64         
 6   Customer Gender_Male                              8469 non-null   int64         
 7   Customer Gender_Other                             8469 non-null   int64         
 8   Product Purchased_Adobe Phot

**Explanation:**
- After date conversion and feature extraction, again checked for missing values.
- The missing values were filled using the median method as it is an continuous variables.
- After the missing value process, the final columns became "8469 non-null", Hence, the dataset is complete and ready for modeling

In [10]:
#drop the original datetime columns after extracting them
data = data.drop(['Date of Purchase','First Response Time','Time to Resolution'], axis=1)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 68 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Customer Age                                      8469 non-null   int64  
 1   Customer Satisfaction Rating                      8469 non-null   float64
 2   Customer Gender_Female                            8469 non-null   int64  
 3   Customer Gender_Male                              8469 non-null   int64  
 4   Customer Gender_Other                             8469 non-null   int64  
 5   Product Purchased_Adobe Photoshop                 8469 non-null   int64  
 6   Product Purchased_Amazon Echo                     8469 non-null   int64  
 7   Product Purchased_Amazon Kindle                   8469 non-null   int64  
 8   Product Purchased_Apple AirPods                   8469 non-null   int64  
 9   Product Purchased_A

**Explanation**
- After extracting the time-based features,then proceeded to remove the original datetime columns - "Date of Purchase,First Response Time,Time to Resolution".
- As this reduced the dataset from 71 to 68 columns.
- After this process, the dataset only contain numeric columns,were now the dataset is cleaned and ready for model processing.

In [11]:
data.to_csv('customer_support_updated.csv')

# Divide data to x,y

In [12]:
# splitting the feature (x) & target variable(y) and checking their shape dimensions
x = data.drop('Resolution Hours', axis=1)
y = data['Resolution Hours']
print(x.shape)
print(y.shape)

(8469, 67)
(8469,)


**Explanation:**
- The dataset was divided into input features (x) and the target variable (y).
- The input features are used by the models to learn patterns in the data.
- Then used data.drop('y', axis=1) to create X and stored the target variable separately as y.
- The shapes were - x.shape = (8469,67); y.shape = (8469,)
- This means there are 8469 records, 67 input features and 1 target variable.

# Data Scaling

In [13]:
from sklearn.preprocessing import StandardScaler
x_scaled = StandardScaler().fit_transform(x)
DataFrame(x_scaled)

,0,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
0,-0.786312,0.003523,-0.719165,-0.720866,1.467316,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,-0.590941,-0.569306,-0.582032,-0.580032,1.730008,-1.007240,0.736934,0.527176,-0.803014
1,-0.132512,0.003523,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,-0.590941,1.756526,-0.582032,-0.580032,-0.578032,-0.429833,0.736934,1.057185,1.233884
2,0.259767,0.003523,-0.719165,-0.720866,1.467316,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,1.762154,-0.590941,-0.569306,-0.582032,-0.580032,1.730008,0.147573,-0.177395,0.394674,-0.870911
3,-1.113212,0.003523,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,1.762154,-0.590941,-0.569306,-0.582032,-0.580032,1.730008,1.302386,-0.291686,-0.135334,0.147538
4,1.501986,-2.482805,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,1.762154,-0.590941,-0.569306,1.718120,-0.580032,-0.578032,-1.295943,-1.320306,-1.062849,-1.006704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8464,-1.440111,0.003523,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,1.762154,-0.590941,-0.569306,-0.582032,1.724044,-0.578032,1.591089,-0.863142,-0.930346,-0.395635
8465,-1.113212,0.003523,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,-0.590941,-0.569306,1.718120,-0.580032,-0.578032,-1.295943,0.736934,-0.930346,-0.395635
8466,0.848187,0.003523,1.390502,-0.720866,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,-0.590941,-0.569306,-0.582032,-0.580032,1.730008,0.436276,0.165479,0.129670,1.165987
8467,0.652047,0.003523,-0.719165,1.387220,-0.681517,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,1.692216,-0.569306,1.718120,-0.580032,-0.578032,1.013683,0.051187,1.322189,0.079642


**Explanation:**
- Then, applied StandardScaler to scale the input features.
- This transforms the values so they have a common scale.
- Scaling is important for SVR because SVR is sensitive to feature magnitude.
- Only the feature matrix x was scaled. The target y was kept in the original unit, which is hours.

# Data Splitting

In [42]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_scaled,y, test_size=0.2, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(6775, 68)
(1694, 68)
(6775,)
(1694,)


**Explanation:**
- Then splitted the data into training and testing sets.
- Then used 80% for training and 20% for testing.
- The output shapes were - x_train = (6775, 67);x_test = (1694, 67);y_train = (6775,);y_test = (1694,).
- This step is important because the model should be trained on one part and evaluated on unseen data

# **Linear Regression**

**Modelling**

In [43]:
from statsmodels.api import OLS, add_constant
x_scaled = add_constant(x_scaled)
DataFrame(x_scaled)
model = OLS(y,x_scaled)
best_model = model.fit()
print(best_model.summary())

                            OLS Regression Results                            
Dep. Variable:       Resolution Hours   R-squared:                       0.492
Model:                            OLS   Adj. R-squared:                  0.488
Method:                 Least Squares   F-statistic:                     133.3
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        14:05:07   Log-Likelihood:                -24210.
No. Observations:                8469   AIC:                         4.854e+04
Df Residuals:                    8407   BIC:                         4.898e+04
Df Model:                          61                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.3436      0.046    224.755      0.0

**Explanation:**
- Then applied Linear Regression using OLS.
- This model helps to find the linear relationship between the input variables and the target variable.
- The most important output here was the R-squared value.
- "The Linear Regression model gave:R² = 0.492"
- This means the model explains about 49.2% of the variation in resolution time.

In [16]:
# checking coefficients
best_model.params

,0
const,10.343581
x1,0.024776
x2,-0.037976
x3,-0.067854
x4,-0.012121
...,...
x63,0.026585
x64,0.071339
x65,-0.030019
x66,-1.737353


**Explanation**
- This shows the intercept and coefficients of the Linear Regression model.
- Each value represents how much a feature affects the prediction.
- Positive values increase prediction, negative values decrease it.

#### **regression equation**
y = B0 + B1*x1 + B2*x2 + ... + B67*x67

Resolution Hours = 10.34 + (0.02 * x1) + (-0.03 * x2) + ... + (-1.73 * x67)

**Explanation**
- This is the Linear Regression equation used for prediction.
- 10.34 is the base value (intercept).
- Remaining terms show contribution of each feature.

In [21]:
import numpy as np
single_input = x_scaled[0]
print(best_model.predict(single_input))

[7.7130379]


**Explanation**

- This predicts resolution time for one new input.
- 1 is added for intercept.
- Remaining values are scaled features.

In [22]:
# making prediction on sample rows (prediction for multiple rows)
best_model.predict(x_scaled[10:19])

array([ 6.62474922,  8.02873177,  5.67925989,  5.86350357,  8.84262089,
        5.99824035,  6.39101969,  5.53557286, 16.62317411])

**Explanation:**
- After fitting the model, proceeded to check the regression coefficients.
- Then, generated the sample predictions using a few rows.
- The predicted values were - 6.62;8.03;5.68;5.86;8.84;5.99;6.39;5.54;16.62.
- These predictions are in hours, which is required for this regression problem.

# **Support Vector Regressor**

# **Method - 1**

**Modelling**

In [23]:
# importing SVR
from sklearn.svm import SVR
SV_regressor1 = SVR(kernel='rbf', C=10, epsilon=0.5) # building SVR model
SV_regressor1.fit(x_train,y_train) # tarining model
y_pred1 = SV_regressor1.predict(x_test) # predicting on test set

**Explanation:**
- Then, the implemented Support Vector Regression (SVR) as the second regression model.
- Used: kernel = 'rbf';C = 10;epsilon = 0.5.
- The RBF kernel helps to capture the non-linear patterns in the data.
- The model was trained on the training data and tested on the test data

**Evaluation**

In [24]:
# import evaluation metric
from sklearn.metrics import r2_score
R2 = r2_score(y_test,y_pred1)
print('R2 = ', round(R2*100,2),'%') #calculating R-squared value

R2 =  51.35 %


**Explanation:**
- Then, used R-squared (R²) to evaluate the regression model.
- The basic SVR model gave - "R² = 51.35%".
- This means the model explains about 51.35% of the variation in the target variable.
- This result was slightly better than the Linear Regression result

# **Method - 2** - GridSearchCV

In [25]:
#import GridSearchCV
from sklearn.model_selection import GridSearchCV
SV_Regressor2 = SVR() # defining model
k_c_e = {'kernel':['linear','poly','rbf','sigmoid'], 'C':[1,10],'epsilon':[0.1,1,]} # defining para grid

grid_search =GridSearchCV(estimator=SV_Regressor2, param_grid=k_c_e, scoring= 'r2', cv=3) # building: 4 componenets
grid_search.fit(x_scaled,y)# fitting: 4 tasks

best_k_c_e = grid_search.best_params_
print(best_k_c_e)
R2 = grid_search.best_score_
print('R2=' ,round(R2*100,2))
best_svr = grid_search.best_estimator_
print(best_svr)

{'C': 10, 'epsilon': 0.1, 'kernel': 'rbf'}
R2= 53.33
SVR(C=10)


**Explanation:**
- Used GridSearchCV to tune the SVR model.
- The parameter values tested were:
kernel = ['linear', 'poly', 'rbf', 'sigmoid'];C = [1, 10];epsilon = [0.1, 1].
- For Cross-validation used - cv = 3.
- The best parameters found were - {'C': 10, 'epsilon': 0.1, 'kernel': 'rbf'}
- The tuned SVR model gave "R² = 53.33"
- This was the best result among all models.

# **Prediction**

In [26]:
# selecting one new sample from input data
new = x.iloc[[0]]
new

,Customer Age,Customer Satisfaction Rating,Customer Gender_Female,Customer Gender_Male,Customer Gender_Other,Product Purchased_Adobe Photoshop,Product Purchased_Amazon Echo,Product Purchased_Amazon Kindle,Product Purchased_Apple AirPods,Product Purchased_Asus ROG,...,Ticket Priority_Low,Ticket Priority_Medium,Ticket Channel_Chat,Ticket Channel_Email,Ticket Channel_Phone,Ticket Channel_Social media,Purchase Month,Purchase Day,First Response Hour,First Response Minute
0,32,3.0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,3,22,12,15


**Explanation**

- Selected one sample row from the input data as a new case.
- Then, used to test how the trained models predict for a single new input.

**Scaling New Input**

In [27]:
# scaling and viewing the new sample using same scaler
new_scaled = StandardScaler().fit(x).transform(new)
DataFrame(new_scaled)

,0,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
0,-0.786312,0.003523,-0.719165,-0.720866,1.467316,-0.14778,-0.16369,-0.154723,-0.160622,-0.150263,...,-0.567487,-0.590941,-0.569306,-0.582032,-0.580032,1.730008,-1.00724,0.736934,0.527176,-0.803014


**Explanation**
- Before prediction, scaled the new input using the same scaler used for training data.
- This ensures that consistency between the training data and new sample data.

**Prediction Using Basic SVR and Tuned SVR**

In [28]:
# prediction using basic SVR
SV_regressor1.predict(new_scaled)

array([5.95027276])

In [32]:
# prediction using tuned SVR
# The best_svr model was trained on x_scaled *after* add_constant was applied for OLS.
# Therefore, it expects 68 features (67 original + 1 constant).
# We need to add a constant term to new_scaled to match this.
import numpy as np
new_scaled_with_constant = np.insert(new_scaled, 0, 1, axis=1)
best_svr.predict(new_scaled_with_constant)

array([6.12363287])

**Explanation:**
- The prediction from the basic SVR model was - "5.95027276 hours".
- The prediction from the tuned SVR model was - "6.14684949 hours".
- These values show that the estimated time needed to resolve the support ticket.

**Prediction Using Linear Regression**

In [33]:
# prediction using linear regression
import numpy as np
new_scaled = np.insert(new_scaled,0,1)

In [34]:
best_model.predict(new_scaled)

array([7.7130379])

**Explanation:**
- Then, predicted the same new sample using Linear Regression.
- The predicted resolution time from Linear Regression was - "7.7130379 hours"
- This helps to compare the prediction behaviour of different models.

 # **Hyperparameter Tuning**

**Explanation**

- To improve the SVR model, performed  the hyperparameter tuning using GridSearchCV.
- The parameters tuned were - kernel;C;epsilon.
- Then best combination found was:kernel = rbf;C = 10;epsilon = 0.1.
- This improved the model performance from 51.35% in the basic SVR model to 52.47% in the tuned SVR model.

 # **Evaluation Metrics**

**Explanation**

- Since this is a regression problem, used R-squared (R²) as the performance metric.
- R² measures how much of the variation in the target variable is explained by the model.
- Hence, the model results were: "Linear Regression R² = 0.492;Basic SVR R² = 51.35%;Tuned SVR R² = 52.47%.
- So, the tuned SVR performed best.

 # **Overfitting Avoidance**

**Explanation**

- Overfitting happens when a model performs well on training data but poorly on unseen data.
- To reduce overfitting in this model -
      - I split the data into training and testing sets.
      - I used cross-validation in GridSearchCV
      - I tested the model on unseen test data
- These steps maked the model more reliable and help to improve generalization.

 # **Result Analysis**

**Explanation**

- The Linear Regression model gave R² = 0.492, which means it explained around 49.2% of the target variation.
- The basic SVR model improved the result and gave R² = 51.35%.
- After tuning, the SVR model gave the highest score, R² = 52.47%.
- This shows that SVR was slightly better than Linear Regression for this dataset.
- So, SVR is more suitable for predicting ticket resolution time.

 # **Deployment Recommendation**

**Explanation**

- Based on the results, the tuned SVR model is recommended for deployment.
- It provided the highest performance among all the models tested.
- The tuned SVR model gave - "R² = 52.47%".
- So it is the most suitable model for predicting customer support ticket resolution time.

 # **Final Conclusion**

**Explanation**

- In conclusion, the dataset was prepared using several preprocessing steps such as: "missing value handling,
removing unwanted columns,one-hot encoding,date conversion,feature engineering,scaling".
- Then two regression models were implemented: "Linear Regression,Support Vector Regression.
- Hyperparameter tuning was also applied using GridSearchCV.
- The final model results were:
        - Linear Regression R² = 0.492
        - Basic SVR R² = 51.35%
        - Tuned SVR R² = 52.47%
- Among all the models, the tuned SVR model achieved the best performance.
- Therefore, tuned SVR is selected as the final model for predicting customer support ticket resolution time.